# meta.json 방출기 테스트 (작업지시서 §4.1)

지금까지 만든 세 산출물이 엑셀의 **내용**이었다면
(칸 값 `cells.jsonl`, 칸끼리 연결 `references.json`, 건강검진표 `diagnostics.json`),
이번 `meta.json`은 패키지의 **표지**입니다.

"이 스킬 패키지가 **어느 원본에서·어떻게** 나왔는가"를 한 장에 적습니다 —
원본 파일명과 지문(sha256)·크기·형식, 어느 로더로 열었나, 시트 목록과 범위,
그리고 만든 시각.

**중요한 규칙 하나**: `meta.json`은 결정론 계층에서 **유일하게 시각(`generated_at`)이
매번 바뀌어도 되는** 파일입니다(§4.1). 그래서 재현성 검증(V3)은 이 한 필드만
빼고 비교합니다. 나머지 값은 같은 파일이면 언제 돌려도 똑같아야 합니다.

확인하는 것:
1. **감사계약 표지** — 필드가 스키마대로 다 채워지는지
2. **원본 지문** — sha256이 파일을 직접 해시한 값과 일치하는지 + 크기·형식
3. **`generated_at` 계약** — 주입/자동, 그리고 이 필드만 빼면 재현되는지
4. **`converter_version` 출처** — pyproject를 유일 출처로 읽는지
5. **결정론 + 32종 전수** — `generated_at` 빼고 두 번 만들면 똑같은가

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비 — 경로 자동 판별과 import
from pathlib import Path
import copy, hashlib, json, sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():   # tests/ 안에서 열었을 때
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.extractor import extract_workbook
from excel_to_skill.meta import build_meta, write_meta, _converter_version

RAW = ROOT / "workingpaper_raw"   # 실조서 코퍼스 (git 밖)
OUT = ROOT / "tests" / "_output"  # 노트북 산출물 (git 밖)
OUT.mkdir(exist_ok=True)
print("프로젝트 루트:", ROOT)
print("코퍼스 파일 수:", len(list(RAW.glob("*.xls*"))))

프로젝트 루트: /home/shin/Project/Excel_to_Skill
코퍼스 파일 수: 32


## 1. 감사계약 표지 — 필드가 스키마대로 다 채워지나

§4.1 스키마의 필드가 전부 제자리에 있는지 봅니다. 시트 목록은 길 수 있어
앞 3개만 보여주고 총 매수만 적습니다. `generated_at`은 재현 비교를 위해
고정값(`2026-07-09T00:00:00Z`)을 주입합니다.

In [2]:
p = next(RAW.glob("*1100~1300*"))
ir = extract_workbook(p)
doc = write_meta(ir, OUT / "감사계약.meta.json", generated_at="2026-07-09T00:00:00Z")

show = copy.deepcopy(doc)
show["sheets"] = show["sheets"][:3] + [f"...(총 {len(doc['sheets'])}매)"]
print(json.dumps(show, ensure_ascii=False, indent=2))

checks = {
    "tool == excel_to_skill":        doc["tool"] == "excel_to_skill",
    "converter_version 채워짐":       bool(doc["converter_version"]),
    "source 키 4종":                  set(doc["source"]) == {"filename","sha256","size_bytes","format"},
    "format == xlsx":                 doc["source"]["format"] == "xlsx",
    "loader_path 채워짐":             bool(doc["loader_path"]),
    "sheets 비지 않음":               len(doc["sheets"]) > 0,
    "sheet 키 4종":                   set(doc["sheets"][0]) == {"name","dimensions","max_row","max_col"},
    "generated_at 주입값 반영":        doc["generated_at"] == "2026-07-09T00:00:00Z",
    "annotation 미주석 고정":          doc["annotation"] == {"present": False, "annotator_version": None, "review_status": None},
}
print("\n── 판정 ──")
for k, v in checks.items():
    print(f"  {'O' if v else 'X'} {k}")
assert all(checks.values()), "필드 판정 실패"
print("결과:", "통과" if all(checks.values()) else "실패")

{
  "tool": "excel_to_skill",
  "converter_version": "0.1.0",
  "source": {
    "filename": "감사조서서식_1100~1300 감사계약.xlsx",
    "sha256": "9c7aa730172a7c9aa1079f5b513accc4856d4ca831ef25ba2bbaf7455daba64e",
    "size_bytes": 57107,
    "format": "xlsx"
  },
  "loader_path": "openpyxl_normal",
  "sheets": [
    {
      "name": "1100",
      "dimensions": "A1:F13",
      "max_row": 13,
      "max_col": 6
    },
    {
      "name": "1200",
      "dimensions": "A1:F35",
      "max_row": 35,
      "max_col": 6
    },
    {
      "name": "1300",
      "dimensions": "A1:F16",
      "max_row": 16,
      "max_col": 6
    },
    "...(총 3매)"
  ],
  "generated_at": "2026-07-09T00:00:00Z",
  "annotation": {
    "present": false,
    "annotator_version": null,
    "review_status": null
  }
}

── 판정 ──
  O tool == excel_to_skill
  O converter_version 채워짐
  O source 키 4종
  O format == xlsx
  O loader_path 채워짐
  O sheets 비지 않음
  O sheet 키 4종
  O generated_at 주입값 반영
  O annotation 미주석 고정
결과: 통과


## 2. 원본 지문 — sha256이 파일 해시와 일치하나

`meta.json`의 `source.sha256`은 원본 파일 바이트를 그대로 해시한 값입니다.
같은 파일을 노트북에서 직접 해시해 대조하고, 크기·형식도 확인합니다.
이 지문이 나중에 "이 패키지가 정말 이 원본에서 나왔는가"를 증명하는 근거가 됩니다.

In [3]:
real_sha = hashlib.sha256(p.read_bytes()).hexdigest()
real_size = p.stat().st_size
print("파일명       :", doc["source"]["filename"])
print("sha256(문서) :", doc["source"]["sha256"])
print("sha256(직접) :", real_sha)
print("크기(문서/직접):", doc["source"]["size_bytes"], "/", real_size, "bytes")
print("형식         :", doc["source"]["format"], "| loader:", doc["loader_path"])
assert doc["source"]["sha256"] == real_sha and len(real_sha) == 64
assert doc["source"]["size_bytes"] == real_size > 0
print("\n지문 대조: 일치 (64자)")

파일명       : 감사조서서식_1100~1300 감사계약.xlsx
sha256(문서) : 9c7aa730172a7c9aa1079f5b513accc4856d4ca831ef25ba2bbaf7455daba64e
sha256(직접) : 9c7aa730172a7c9aa1079f5b513accc4856d4ca831ef25ba2bbaf7455daba64e
크기(문서/직접): 57107 / 57107 bytes
형식         : xlsx | loader: openpyxl_normal

지문 대조: 일치 (64자)


## 3. `generated_at` 계약 — 주입/자동, 그리고 이 필드만 빼면 재현되나

- `generated_at`을 주면 그 값이 그대로 들어갑니다(테스트·비교용).
- 안 주면 실행 시각(UTC, `...Z`)이 자동으로 들어갑니다.
- **핵심**: 두 번 만들면 `generated_at`만 다르고 **나머지는 전부 같아야** 합니다.
  V3 재현성 검증이 정확히 이 방식(이 필드 제외 정규화 비교)입니다.

In [4]:
auto1 = build_meta(ir)           # 자동 시각
auto2 = build_meta(ir)           # 다시 자동 시각
print("자동 generated_at 예시:", auto1["generated_at"])
print("'...Z'로 끝남:", auto1["generated_at"].endswith("Z"))

def norm(d):
    d = {**d}; d.pop("generated_at"); return json.dumps(d, ensure_ascii=False)

same_except_ts = norm(auto1) == norm(auto2)
print("generated_at 제외 두 산출물 동일:", same_except_ts)
assert same_except_ts, "generated_at 외 값이 흔들림 — 결정론 위반"
print("결과: 통과 (가변 필드는 generated_at 하나뿐)")

자동 generated_at 예시: 2026-07-09T02:38:52Z
'...Z'로 끝남: True
generated_at 제외 두 산출물 동일: True
결과: 통과 (가변 필드는 generated_at 하나뿐)


## 4. `converter_version` 출처 — pyproject를 유일 출처로 읽나

버전은 코드에 하드코딩하지 않고 설치 메타데이터(pyproject `[project].version`)에서
읽습니다. 이중 관리를 없애기 위한 결정입니다. 지금 값은 `0.1.0`이어야 합니다.

In [5]:
ver = _converter_version()
print("converter_version =", ver)
# pyproject의 값과 대조
import re
pyproj = (ROOT / "pyproject.toml").read_text(encoding="utf-8")
m = re.search(r'^version\s*=\s*"([^"]+)"', pyproj, re.M)
print("pyproject version =", m.group(1))
assert ver == m.group(1), "설치 메타데이터와 pyproject 불일치"
print("결과: 통과 (유일 출처 = pyproject)")

converter_version = 0.1.0
pyproject version = 0.1.0
결과: 통과 (유일 출처 = pyproject)


## 5. 결정론 + 32종 전수

코퍼스 32종 전부를 열어 표지를 만들고, `generated_at`만 빼면
두 번 만든 결과가 sha256까지 같은지(결정론) 봅니다. 형식·로더·시트 수도 한눈에.

In [6]:
def digest(d):
    n = {**d}; n.pop("generated_at")
    return hashlib.sha256(json.dumps(n, ensure_ascii=False).encode()).hexdigest()

print(f"{'파일':<40} {'fmt':<5} {'loader':<30} {'시트':>4} 재현")
fails = []
for fp in sorted(RAW.glob("*.xls*")):
    try:
        d1 = build_meta(extract_workbook(fp), generated_at="A")
        d2 = build_meta(extract_workbook(fp), generated_at="B")
        same = digest(d1) == digest(d2)
        if not same:
            fails.append((fp.name, "결정론 불일치"))
        print(f"{fp.name[:38]:<40} {d1['source']['format']:<5} "
              f"{d1['loader_path']:<30} {len(d1['sheets']):>4} {'O' if same else 'X'}")
    except Exception as e:
        fails.append((fp.name, repr(e)))
        print(f"{fp.name[:38]:<40} 실패: {e!r}")

print(f"\n실패 {len(fails)}")
for name, err in fails:
    print("  X", name, err)
assert not fails, "전수 실패 존재"
print("32종 전수: 실패 0 · 결정론 재현 O")

파일                                       fmt   loader                           시트 재현
감사조서서식_01 조서파일 KIFRS.xlsx                xlsx  openpyxl_normal                   3 O


감사조서서식_02 영구조서목록.xlsx                    xlsx  openpyxl_normal                   1 O


감사조서서식_03 일반조서목록 KIFRS.xlsx              xlsx  openpyxl_normal                   1 O
감사조서서식_04 감사조서철 작성 및 보존.xlsx             xlsx  openpyxl_normal                   1 O


감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx      xlsx  openpyxl_normal                   1 O


감사조서서식_1100~1300 감사계약.xlsx               xlsx  openpyxl_normal                   3 O


...

감사조서서식_7511A_그룹감사결론을 위한 분석적절차_2025 신규.   xlsx  openpyxl_normal                   1 O


감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xl   xls   xlrd                              5 O
감사조서서식_7555 연결심리사항점검표.xlsx               xlsx  openpyxl_normal                   1 O
감사조서서식_7555A_그룹감사 업무수행이사와 논의한 사항 및 자문한   xlsx  openpyxl_normal                   1 O


감사조서서식_8100~8700 감사완결 2025.xlsx          xlsx  openpyxl_normal                  12 O


감사조서서식_8400 공시사항점검표 (KIFRS용)_2025.xls    xls   xlrd                              5 O
감사조서서식_8550 심리사항점검표.xlsx                 xlsx  openpyxl_normal                   1 O
감사조서서식_8550A_업무수행이사와 논의한 사항 및 자문한 사항 2   xlsx  openpyxl_normal                   1 O


감사조서서식_9100~9800 내부회계관리제도감사 조서서식(예시).x   xlsx  openpyxl_read_only+xml_merge     25 O

실패 0
32종 전수: 실패 0 · 결정론 재현 O
